<img src="logo.png" alt="Vegeta" width="240">

# Submarine — a small autonomous underwater vehicle: buoyancy and trim, resistance (CFD), propulsion and endurance, pressure hull (FEA), the propeller (CFD, noise, blade modes), diving, depth cycling

A 1.2 m, 180 mm diameter AUV: Myring body of revolution, sail, cruciform stern fins, and inside it a
cylindrical aluminium pressure hull with domed ends. Everything is a parameter of
`designs/submarine.py`; the `part` parameter selects the outer body (hydrostatics, CFD), the pressure
hull (FEA) or the whole vehicle.

```
CAD (Dedalus): vehicle, body and pressure hull ─► STEP/STL, volumes, centres
mass budget ─► buoyancy, trim lead and its position, BG (submerged stability)
resistance: ITTC friction + form factor + appendages ─► speed–power curve
CFD (Aeromant rans_ksst_external, OpenFOAM): fully submerged drag at cruise against the estimate
propulsion (Boreas in water): thruster operating points, top speed, endurance and range vs speed
FEA (Talos): pressure hull at the design depth — stress vs thin-shell theory, collapse pressures
propeller: CAD ─► rotor CFD (thrust, torque, efficiency vs BEMT, a video) ─► noise and cavitation ─► blade stress (a video), modes, Campbell
diving: speed and depth over a profile dive, pressure and hull stress with depth, a propulsion loss
depth cycling (Chronos): three dive profiles ─► pressure spectra ─► hull fatigue damage per dive
export: one JSON
```

Every material, depth and mass is an explicit input. The OpenFOAM cells run when you run them
(`VEGETA_SKIP_OPENFOAM=1` skips them in headless execution).

In [ ]:
import json, math, os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from vegeta import dedalus, talos, aeromant, boreas, chronos
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.aeromant import viz as aviz
from vegeta.dedalus.examples import Propeller as PropellerCAD

RUNS = Path("_runs/submarine"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
design_file = RUNS / "submarine.py"
shutil.copy(Path("designs/submarine.py"), design_file)
sub_design = dedalus.load_design(f"{design_file}:Submarine")
p = sub_design.params.defaults
RHO_W, NU_W, G = 1025.0, 1.05e-6, 9.81            # sea water
DESIGN_DEPTH_M = 200.0                             # rated depth (engineering input)
TEST_FACTOR = 1.5                                  # test depth = 1.5 x rated depth
V_CRUISE = 1.5                                     # m/s
pd.DataFrame(sub_design.params.table()).set_index("name")

## 1. The vehicle (Dedalus)

The pressure hull is modelled in its own frame (cylinder from x = 0 to 500 mm); it sits in the parallel
middle body of the vehicle, offset by `X_PH`.

In [ ]:
vehicle = sub_design.generate()
body = sub_design.generate(part="body")
hull = sub_design.generate(part="pressure_hull")
X_PH = 420.0                                       # pressure hull cylinder start, from the tail tip [mm]
hull_in_place = dedalus.Geometry.from_cadquery(hull.shape.translate((X_PH, 0, 0)), name="pressure_hull")
mv, mb, mh = vehicle.measure(), body.measure(), hull.measure()
files = vehicle.export(RUNS / "cad", formats=("step", "stl"), stl_tolerance=0.2)
print(f"vehicle {mv['volume'] * 1e-6:.2f} L, wetted surface {mv['surface_area'] * 1e-6:.3f} m^2 | bare body {mb['volume'] * 1e-6:.2f} L, "
      f"{mb['surface_area'] * 1e-6:.3f} m^2 | pressure hull shell {mh['volume'] * 1e-6:.2f} L")
dviz.show(dviz.plot3d(vehicle, hull_in_place, opacity=0.45))
fig = dviz.plot_sections(vehicle, normal="x", positions=[170.0, 400.0, 660.0, 1000.0], cols=4)

## 2. Mass, buoyancy and trim

Fully submerged, buoyancy is fixed by the vehicle volume. The budget lists every mass with its position
(x from the tail tip, z from the axis, both in mm); the trim lead is computed to leave the vehicle
`RESERVE` positively buoyant with the centre of gravity under the centre of buoyancy. For a submerged
body the stability lever is simply BG.

In [ ]:
RESERVE = 0.005                                    # 0.5 % positive buoyancy: it surfaces when power is lost
buoyancy_kg = mv["volume"] * 1e-9 * RHO_W
CB = np.array(mv["center_of_mass"])
RHO_AL, RHO_GFRP, FAIRING_T = 2.70e-6, 1.8e-6, 2.0   # kg/mm^3, kg/mm^3, mm
items = pd.DataFrame([
    ("pressure hull (6061-T6, from CAD)", mh["volume"] * RHO_AL, X_PH + mh["center_of_mass"][0], 0.0),
    ("fairing, sail and fins (GFRP 2 mm)", mv["surface_area"] * FAIRING_T * RHO_GFRP, CB[0], CB[2]),
    ("battery 7S Li-ion 20 Ah", 6.0, 560.0, -30.0),
    ("electronics, INS, acoustic modem", 1.5, 780.0, 0.0),
    ("sonar payload (nose, free flooded)", 3.5, 1060.0, -15.0),
    ("thruster, propeller, stern gear", 1.5, 80.0, 0.0),
], columns=["item", "mass_kg", "x_mm", "z_mm"]).set_index("item")
M_target = buoyancy_kg * (1 - RESERVE)
m_lead = M_target - items["mass_kg"].sum()
if m_lead < 0:
    raise ValueError(f"the vehicle is {-m_lead:.2f} kg too heavy for its volume: enlarge the hull or cut the budget")
x_lead = (CB[0] * M_target - (items["mass_kg"] * items["x_mm"]).sum()) / m_lead
items.loc["trim lead (computed)"] = [m_lead, x_lead, -60.0]
M_KG = items["mass_kg"].sum()
CG = np.array([(items["mass_kg"] * items["x_mm"]).sum() / M_KG, 0.0, (items["mass_kg"] * items["z_mm"]).sum() / M_KG])
BG = CB[2] - CG[2]
hydro = pd.Series({"buoyancy_kg": buoyancy_kg, "mass_kg": M_KG, "net_buoyancy_N": (buoyancy_kg - M_KG) * G, "LCB_mm": CB[0], "LCG_mm": CG[0],
                   "BG_mm": BG, "righting_moment_at_90deg_Nm": M_KG * G * BG / 1000, "trim_lead_kg": m_lead, "trim_lead_x_mm": x_lead})
print(f"the lead goes {x_lead:.0f} mm from the tail ({'inside the pressure hull' if X_PH < x_lead < X_PH + p['pressure_hull_length'] else 'outside the pressure hull'})")
display(items.round(2)); hydro.round(3)

## 3. Resistance

Deeply submerged there is no wave drag: friction on the wetted surface (ITTC 1957 line) with a form
factor for the body of revolution (Hoerner, `k = 1.5 (D/L)^1.5 + 7 (D/L)^3`), plus the appendages
(sail and fins) at 1.5 × friction with an interference allowance.

In [ ]:
L_M, D_M = p["length"] / 1000, p["diameter"] / 1000
S_WET, S_BODY = mv["surface_area"] * 1e-6, mb["surface_area"] * 1e-6
def resistance(V):
    Re = V * L_M / NU_W
    Cf = 0.075 / (math.log10(Re) - 2) ** 2
    k = 1.5 * (D_M / L_M) ** 1.5 + 7 * (D_M / L_M) ** 3
    R_body = 0.5 * RHO_W * V**2 * S_BODY * Cf * (1 + k)
    R_app = 0.5 * RHO_W * V**2 * (S_WET - S_BODY) * Cf * 1.5 * 1.3
    return R_body + R_app, R_body, R_app, Cf, k
Vs = np.linspace(0.3, 4.0, 38)
R = np.array([resistance(v)[:3] for v in Vs])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(Vs, R[:, 0], label="total"); ax[0].plot(Vs, R[:, 1], "--", label="bare body"); ax[0].plot(Vs, R[:, 2], ":", label="appendages")
ax[0].set(xlabel="speed [m/s]", ylabel="resistance [N]"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(Vs, R[:, 0] * Vs); ax[1].set(xlabel="speed [m/s]", ylabel="effective power R·V [W]"); ax[1].grid(alpha=0.3)
R_cruise, _, _, Cf_c, k_form = resistance(V_CRUISE)
print(f"at {V_CRUISE} m/s: Re {V_CRUISE * L_M / NU_W:.2e}, Cf {Cf_c:.4f}, form factor 1+k = {1 + k_form:.3f}, resistance {R_cruise:.2f} N, effective power {R_cruise * V_CRUISE:.1f} W")

### CFD check at cruise (OpenFOAM, fully submerged)

`rans_ksst_external` (k-ω SST, steady) on the whole vehicle in water: no free surface, so a plain
external-flow case. The free stream is along +x, so the CAD is turned round (the nose must point
upstream). The template scales its domain with `reference_length` and accepts bodies up to 4 × that
length, hence L/4 here; `reference_area` is the wetted surface, so `Cd` compares directly with
`Cf (1+k)`. Coarse settings: expect the drag within a few tens of percent of the estimate.

In [ ]:
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
nose_upstream = dedalus.Geometry.from_cadquery(vehicle.shape.rotate((0, 0, 0), (0, 0, 1), 180), name="vehicle_nose_upstream")
stl_cfd = nose_upstream.export_stl(RUNS / "cad" / "vehicle_nose_upstream.stl", tolerance=0.2)
CFD_PARAMS = dict(velocity=V_CRUISE, kinematic_viscosity=NU_W, density=RHO_W, reference_area=S_WET, reference_length=L_M / 4,
                  center_of_rotation=(-CB[0] / 1000, 0.0, 0.0), iterations=400, residual_target=1e-4,
                  cells_per_length=3.0, surface_level=4, near_level=3, wake_level=2)
cfd = None
if RUN_CFD:
    case = aeromant.CFDCase("rans_ksst_external", stl_cfd, CFD_PARAMS, workdir=RUNS / "cfd_cruise", geometry_units="mm",
                            environment=aeromant.OpenFOAMEnvironment.detect())
    print(case.prepare(overwrite=True))
    cfd = case.run(progress=True)
    print(cfd)
else:
    print("CFD skipped (VEGETA_SKIP_OPENFOAM=1): run this cell on a machine with OpenFOAM")

In [ ]:
if cfd is not None and cfd.ok:
    m = cfd.metrics
    compare = pd.DataFrame({"estimate": {"drag_N": R_cruise, "Cd (wetted)": R_cruise / (0.5 * RHO_W * V_CRUISE**2 * S_WET)},
                            "CFD": {"drag_N": m["drag_force_N"], "Cd (wetted)": m["Cd"]}})
    compare["CFD / estimate"] = compare["CFD"] / compare["estimate"]
    display(compare.round(4))
    print("converged:", m["converged"], "| lift (should be ~0 on a symmetric body, the sail adds a little):", round(m["lift_force_N"], 3), "N")
    aviz.show(aviz.plot_field_slice(case, "U", normal="y"))
    fig = aviz.plot_section(case, "p", normal="y", zoom=1.5)

## 4. Propulsion and endurance (Boreas in water)

A 120 mm three-blade propeller on a direct-drive thruster. Wake fraction and thrust deduction are
inputs (the propeller sees `WAKE × V`, and must deliver `R / (1 − T_DED)`). Endurance uses the usable
battery energy over propulsion plus the hotel load; the range curve has the usual optimum where the
hotel load stops dominating.

In [ ]:
WAKE, T_DED, HOTEL_W = 0.85, 0.10, 25.0
prop = boreas.Propeller.from_pitch("120 mm 3-blade", 0.120, 0.100, blades=3, chord_root_m=0.018, chord_max_m=0.030, chord_tip_m=0.012,
                                   mass_kg=0.06, rotor_mass_kg=0.15, notes="generic planform")
section = boreas.Airfoil(name="marine blade section", cl_alpha=5.5, alpha0_deg=-2.0, cl_max=1.0, cd0=0.02, k=0.05, source="assumed")
motor = boreas.Motor("thruster 100KV", kv_rpm_per_volt=100, resistance_ohm=0.6, no_load_current_a=0.3, max_current_a=5.0, mass_kg=0.35)
battery = boreas.Battery("7S Li-ion 20 Ah", cells=7, capacity_ah=20.0, usable_fraction=0.85, mass_kg=6.0)
drive = boreas.Propulsion(prop, section, motor, battery, rho=RHO_W)
def point_at(V):
    return drive.for_thrust(resistance(V)[0] / (1 - T_DED), WAKE * V)
lo, hi = 0.3, 5.0
for _ in range(24):                                            # top speed: full-throttle thrust = resistance
    Vm = 0.5 * (lo + hi)
    lo, hi = (Vm, hi) if drive.at_throttle(1.0, WAKE * Vm).thrust * (1 - T_DED) > resistance(Vm)[0] else (lo, Vm)
V_MAX = lo
cruise, full = point_at(V_CRUISE), drive.at_throttle(1.0, WAKE * V_MAX)
speeds = np.linspace(0.5, V_MAX, 14)
pts = [point_at(v) for v in speeds]
endurance_h = np.array([battery.usable_wh / (pt.electrical_power + HOTEL_W) for pt in pts])
range_km = endurance_h * speeds * 3.6
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(speeds, [pt.electrical_power for pt in pts], label="propulsion (electrical)"); ax[0].axhline(HOTEL_W, ls="--", color="#888", label="hotel load")
ax[0].set(xlabel="speed [m/s]", ylabel="power [W]"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(speeds, range_km); ax[1].set(xlabel="speed [m/s]", ylabel="range [km]"); ax[1].grid(alpha=0.3)
i_best = int(np.argmax(range_km))
print(f"top speed {V_MAX:.2f} m/s at {full.rpm:.0f} rpm, {full.electrical_power:.0f} W{' (current limited)' if full.current_limited else ''} | "
      f"best range {range_km[i_best]:.0f} km at {speeds[i_best]:.2f} m/s | at cruise {V_CRUISE} m/s: {endurance_h[np.argmin(abs(speeds - V_CRUISE))]:.1f} h")
pd.DataFrame({"cruise": {"speed_m_s": V_CRUISE, "rpm": cruise.rpm, "thrust_N": cruise.thrust, "prop_efficiency": cruise.aero.efficiency, "motor_efficiency": cruise.motor_efficiency,
                         "electrical_W": cruise.electrical_power, "current_A": cruise.current},
              "top speed": {"speed_m_s": V_MAX, "rpm": full.rpm, "thrust_N": full.thrust, "prop_efficiency": full.aero.efficiency, "motor_efficiency": full.motor_efficiency,
                            "electrical_W": full.electrical_power, "current_A": full.current}}).round(3)

## 5. The pressure hull at the design depth (Talos)

External pressure `ρ g h` on every wetted face of the shell; the two small flats are the penetrator
plates — the stern one holds the vessel (fixed), the bow one is free and unloaded. Selecting the loaded
faces goes through `inspect_step`: the outer faces are the non-planar ones whose bounding box reaches
the outer radius. Aluminium 6061-T6 as machined (welded properties would be much lower — an input to
change if the hull is welded).

Thin-shell theory for the cylinder: hoop `p r / t`, axial `p r / 2t`, von Mises `0.866 × hoop`; the
FEA should reproduce that away from the ends and show the extra bending where the caps meet the cylinder.

In [ ]:
P_DESIGN = RHO_W * G * DESIGN_DEPTH_M / 1e6                     # MPa
AL = talos.Material("6061-T6", youngs_modulus=69000.0, poissons_ratio=0.33, density=2.7e-9, yield_strength=240.0, source="handbook, machined")
hfiles = hull.export(RUNS / "cad_hull", formats=("step",))
info = talos.inspect_step(hfiles.artifacts["step"], "mm-N-MPa")
R_I, T_SH, D_CAP, L_PH = p["diameter"] / 2 - 12.0, p["shell"], p["end_cap_depth"], p["pressure_hull_length"]   # outer radius of the shell, wall, cap depth, cylinder length
x_flat = D_CAP * math.cos(math.asin(0.12))                    # the flats sit this far beyond the cylinder ends
faces = pd.DataFrame([{"tag": s.tag, "kind": s.kind, "area_mm2": round(s.area), "x_min": round(s.bbox_min[0], 1), "x_max": round(s.bbox_max[0], 1), "r_max": round(s.bbox_max[1], 1)} for s in info.surfaces]).set_index("tag")
wet_tags = [s.tag for s in info.surfaces if s.kind != "Plane" and s.bbox_max[1] > R_I - T_SH / 2]
display(faces); print("wetted (loaded) faces:", wet_tags)
REGIONS = [talos.Surfaces("wet", wet_tags), talos.SurfacesOnPlane("stern_plate", "x", -x_flat, tol=0.05), talos.SurfacesOnPlane("bow_plate", "x", L_PH + x_flat, tol=0.05)]
model = talos.StructuralModel(hfiles.artifacts["step"], "mm-N-MPa", AL, REGIONS, [talos.FixedSupport("stern_plate")], [talos.Pressure("wet", P_DESIGN)],
                              talos.MeshSettings(element_size=6.0), name="design_depth")
mesh_res = model.mesh(RUNS / "hull_fea", progress=True)
res = model.solve(RUNS / "hull_fea", progress=True)
print(f"design depth {DESIGN_DEPTH_M:.0f} m -> {P_DESIGN:.3f} MPa external pressure")
print(res)
tviz.show(tviz.plot_problem(model, mesh_res, arrow_scale=10))

In [ ]:
fr = talos.read_frd(res.artifacts["frd"])
x, y, z = fr.coords.T; r = np.hypot(y, z); vm = fr.von_mises
mid = (x > 0.3 * L_PH) & (x < 0.7 * L_PH)
r_m = R_I - T_SH / 2
hoop = P_DESIGN * r_m / T_SH; axial = hoop / 2; vm_theory = math.sqrt(hoop**2 + axial**2 - hoop * axial)
dr_theory = P_DESIGN * r_m**2 / (AL.youngs_modulus * T_SH) * (1 - AL.poissons_ratio / 2)
u_r = np.abs((fr.displacement[:, 1] * y + fr.displacement[:, 2] * z) / np.maximum(r, 1e-9))
check = pd.DataFrame({"thin-shell theory": {"hoop_MPa": hoop, "axial_MPa": axial, "von_mises_MPa": vm_theory, "radial_displacement_mm": dr_theory},
                      "FEA mid-cylinder (outer/inner mean)": {"hoop_MPa": float("nan"), "axial_MPa": float("nan"),
                                                              "von_mises_MPa": f"{vm[mid & (r > R_I - 0.5)].mean():.2f} / {vm[mid & (r < R_I - T_SH + 0.5)].mean():.2f}",
                                                              "radial_displacement_mm": u_r[mid & (r > R_I - 0.5)].mean()}})
display(check)
loc = res.metrics["max_von_mises_location"]
print(f"peak von Mises {res.metrics['max_von_mises']:.1f} MPa at x = {loc[0]:.0f} mm, r = {math.hypot(loc[1], loc[2]):.0f} mm "
      f"({'inner' if math.hypot(loc[1], loc[2]) < R_I - T_SH / 2 else 'outer'} surface, {'near the bow cap' if loc[0] > L_PH / 2 else 'near the stern cap'}); "
      f"SF to yield {res.metrics['safety_factor_yield']:.1f}")
# collapse pressures of the cylinder (the caps and the plate joints are not covered by these formulas)
D_m = 2 * r_m
p_yield = AL.yield_strength * T_SH / r_m
p_buckle = 2.42 * AL.youngs_modulus * (T_SH / D_m) ** 2.5 / ((1 - AL.poissons_ratio**2) ** 0.75 * (L_PH / D_m - 0.45 * math.sqrt(T_SH / D_m)))
collapse = pd.DataFrame({"pressure_MPa": {"design depth": P_DESIGN, "test depth": TEST_FACTOR * P_DESIGN, "yield (p r / t = σ_y)": p_yield, "elastic buckling (Windenburg–Trilling)": p_buckle}})
collapse["depth_m"] = collapse["pressure_MPa"] * 1e6 / (RHO_W * G)
collapse["factor on design"] = collapse["pressure_MPa"] / P_DESIGN
display(collapse.round(2))
tviz.show(tviz.plot_results(res, field="von_mises", deform_scale=200))
fig = tviz.plot_section(res, normal="z", origin=(0, 0, 0), field="von_mises")

## 6. The propeller: CFD check, noise and cavitation, blade stress and modes

The 120 mm three-blade propeller of section 4 as a solid (the Dedalus `Propeller` example), then:
a rotating-frame CFD case at the cruise point (`rotor_mrf`, sea water) whose thrust, torque and
efficiency are compared with blade element theory; Gutin's blade-passing tones and a broadband
allowance in dB re 1 µPa; the cavitation number at shallow running depth and at the rated depth (deep,
cavitation is not the issue — noise from it near the surface is); one blade under the top-speed load
with its modes, and the Campbell diagram of the blade modes against the shaft and blade-pass lines.

In [ ]:
CAD_KW = dict(diameter=120.0, pitch=100.0, blades=3, hub_diameter=24.0, hub_height=16.0, bore=8.0, chord_root=18.0, chord_max=30.0,
              chord_tip=12.0, thickness=0.12, camber=0.05, stations=8)
prop_cad = PropellerCAD().generate(**CAD_KW)
pfiles = prop_cad.export(RUNS / "cad_prop", formats=("step", "stl"), stl_tolerance=0.02)
print(f"propeller CAD volume {prop_cad.volume:.0f} mm^3 -> {prop_cad.volume * 2.7e-3:.0f} g in aluminium")
dviz.show(dviz.plot3d(prop_cad))
prop_x = dedalus.Geometry.from_cadquery(prop_cad.shape.rotate((0, 0, 0), (0, 1, 0), 90), name="prop_axis_x")   # axis z -> +x
stl_px = prop_x.export_stl(RUNS / "cad_prop" / "prop_axis_x.stl", tolerance=0.02)
PROP_CFD = dict(rpm=cruise.rpm, airspeed=WAKE * V_CRUISE, diameter=CAD_KW["diameter"] / 1000, kinematic_viscosity=NU_W, density=RHO_W, rotation=1,
                iterations=400, cells_per_diameter=6.0, surface_level=3, near_level=2, rotor_level=2, wake_level=1)
cfd_prop = None
if RUN_CFD:
    case_prop = aeromant.CFDCase("rotor_mrf", stl_px, PROP_CFD, workdir=RUNS / "cfd_propeller", geometry_units="mm",
                                 environment=aeromant.OpenFOAMEnvironment.detect())
    print(case_prop.prepare(overwrite=True))
    cfd_prop = case_prop.run(progress=True)
    print(cfd_prop)
else:
    print("propeller CFD skipped (VEGETA_SKIP_OPENFOAM=1): run this cell on a machine with OpenFOAM")

In [ ]:
if cfd_prop is not None and cfd_prop.ok:
    m = cfd_prop.metrics
    compare = pd.DataFrame({"BEMT (Boreas)": {"thrust_N": cruise.thrust, "torque_Nm": cruise.aero.torque, "power_W": cruise.aero.power, "efficiency": cruise.aero.efficiency},
                            "CFD (rotor_mrf)": {"thrust_N": m["thrust_N"], "torque_Nm": m["torque_Nm"], "power_W": m["power_W"], "efficiency": m["efficiency"]}})
    compare["CFD / BEMT"] = compare["CFD (rotor_mrf)"] / compare["BEMT (Boreas)"]
    print(f"{m['mesh_cells']} cells, {'converged' if m['converged'] else 'not converged'} in {m['iterations']} iterations; J = {m['advance_ratio']:.2f}")
    display(compare.round(3))
    aviz.show(aviz.plot_field_slice(case_prop, "U", normal="z"))
    fig = aviz.plot_section(case_prop, "p", normal="z", zoom=2)          # the suction-side pressure minimum is the cavitation indicator
    aviz.show(aviz.plot_surface_pressure(case_prop))
    from IPython.display import Video
    video = aviz.animate_particles(case_prop, RUNS / "propeller_flow.mp4", rpm=cruise.rpm, seconds=6, fps=24)   # tracer particles in the converged field, OpenCV
    display(Video(str(video), embed=False, width=720))

### A particle movie from the OpenFOAM result

A few tracer particles carried through the converged velocity field in water, in a side view and along the axis, the blades turning in consistent slow motion (every frame the rotor turns 10° and the flow advances by the same time). `vegeta.aeromant.movie` reads the case once and draws with OpenCV; the swirl behind the disc is the swirl the rotor puts in.

In [ ]:
if cfd_prop is not None and cfd_prop.ok:
    from IPython.display import Video
    from vegeta.aeromant import movie
    clip = movie.make_movie(case_prop, RUNS / "propeller_particles.mp4", blades=prop.blades, n=40, seconds=12, fps=24, degrees_per_frame=10,
                            title="120 mm AUV propeller at cruise (rotor_mrf, sea water): tracer particles")
    display(Video(str(clip), embed=False, width=900))

### Noise and cavitation

In [ ]:
CP_MIN = -1.0                                  # section minimum pressure coefficient (assumed; a section property)
DIST = 1.0
noise_rows = {}
for name, pt, V in (("cruise", cruise, V_CRUISE), ("top speed", full, V_MAX)):
    tones = boreas.gutin_harmonics(prop, pt.thrust, pt.aero.torque, pt.rpm, DIST, 90.0, boreas.SEA_WATER, harmonics=5)
    bb = boreas.broadband_level(prop, pt.thrust, pt.rpm, DIST, boreas.SEA_WATER)
    total = 10 * math.log10(10 ** (tones["total_tonal_db"] / 10) + 10 ** (bb / 10))
    shallow = boreas.cavitation(prop, pt.rpm, WAKE * V, 5.0, boreas.SEA_WATER, cp_min=CP_MIN)
    deep = boreas.cavitation(prop, pt.rpm, WAKE * V, DESIGN_DEPTH_M, boreas.SEA_WATER, cp_min=CP_MIN)
    noise_rows[name] = {"rpm": pt.rpm, "BPF_hz": tones["blade_pass_hz"], "tonal_dB_re_1uPa_1m": tones["total_tonal_db"], "broadband_dB": bb, "total_dB": total,
                        "sigma_at_5m": shallow["cavitation_number"], "cavitates_at_5m": shallow["cavitates"], "sigma_at_rated_depth": deep["cavitation_number"],
                        "cavitates_at_rated_depth": deep["cavitates"]}
    if name == "cruise":
        fig, ax = plt.subplots(figsize=(6.5, 3.4))
        ax.bar(tones["frequency_hz"], tones["spl_db"], width=3, label="Gutin tones, cruise")
        ax.axhline(bb, color="#c62828", ls="--", label="broadband allowance"); ax.set(xlabel="frequency [Hz]", ylabel="dB re 1 µPa at 1 m"); ax.legend(); ax.grid(alpha=0.3)
noise = pd.DataFrame(noise_rows).T
noise

### One blade at top speed, its modes, and the Campbell diagram

Hub faces fixed; the blade's share of the top-speed thrust and torque (at 0.7 R) as tractions over the
blade; aluminium like the hull. Modes in air × an added-mass factor for water. The video ramps the load
from zero to full while the blade turns at the top-speed rpm — a linear static result in motion.

In [ ]:
blade = PropellerCAD().generate(**dict(CAD_KW, blades=1))
bfiles = blade.export(RUNS / "cad_blade", formats=("step",))
hub_r, hub_h, R_tip = CAD_KW["hub_diameter"] / 2, CAD_KW["hub_height"], CAD_KW["diameter"] / 2
BLADE_REGIONS = [talos.SurfacesInBox("hub", (-hub_r - 0.5, -hub_r - 0.5, -hub_h / 2 - 0.5, hub_r + 0.5, hub_r + 0.5, hub_h / 2 + 0.5)),
                 talos.SurfacesInBox("blade", (hub_r - 2.0, -R_tip, -R_tip, R_tip + 1.0, R_tip, R_tip))]
T_blade = full.thrust / prop.blades                                  # N per blade at top speed
F_tan = full.aero.torque / (prop.blades * 0.7 * prop.radius)         # tangential force per blade at 0.7 R
blade_model = talos.StructuralModel(bfiles.artifacts["step"], "mm-N-MPa", AL, BLADE_REGIONS, [talos.FixedSupport("hub")],
                                    [talos.Force("blade", fz=T_blade, fy=-F_tan)], talos.MeshSettings(element_size=2.0), name="blade_top_speed")
blade_mesh = blade_model.mesh(RUNS / "blade_fea", progress=True)
res_blade = blade_model.solve(RUNS / "blade_fea", progress=True)
print(f"per blade at top speed ({full.rpm:.0f} rpm): thrust {T_blade:.2f} N, tangential {F_tan:.2f} N")
print(res_blade)
shutil.copytree(RUNS / "blade_fea", RUNS / "blade_modal", dirs_exist_ok=True)
blade_modes = blade_model.solve_modes(RUNS / "blade_modal", n_modes=4)
ADDED_MASS_FACTOR = 0.65                                            # in-water / in-air frequency, assumed for a thin blade
f_air = blade_modes.metrics["frequencies_hz"]
f_water = [f * ADDED_MASS_FACTOR for f in f_air]
print("blade modes in air:", [round(f) for f in f_air], "Hz; in water (x0.65):", [round(f) for f in f_water], "Hz")
if res_blade.ok:
    tviz.show(tviz.plot_results(res_blade, field="von_mises"))
    from IPython.display import Video
    video = tviz.animate(res_blade, RUNS / "blade_stress.mp4", rpm=full.rpm, axis="z", seconds=5, fps=24)
    display(Video(str(video), embed=False, width=720))

In [ ]:
blade_structure = chronos.Structure(tuple(f_water), damping_ratio=0.02, source="Talos modal x added-mass factor 0.65")
fig = blade_structure.campbell({"1P": 1, f"{prop.blades}P (blade pass)": prop.blades}, np.linspace(200, full.rpm * 1.2, 30),
                               operating_rpm={"cruise": cruise.rpm, "top speed": full.rpm})
f_ring = math.sqrt(AL.youngs_modulus * 1e6 / (AL.density * 1e12 * (1 - AL.poissons_ratio**2))) / (2 * math.pi * r_m / 1000)   # hull ring (breathing) mode
print(f"pressure hull ring mode ~{f_ring / 1000:.1f} kHz: far above every rotor line")
pd.DataFrame({k: {"BPF_hz": prop.blades * pt.rpm / 60, "nearest_blade_mode_hz": blade_structure.nearest_mode(prop.blades * pt.rpm / 60),
                  "margin": blade_structure.margin(prop.blades * pt.rpm / 60), "amplification": float(blade_structure.amplification(prop.blades * pt.rpm / 60)[0])}
              for k, pt in (("cruise", cruise), ("top speed", full))}).T.round(2)
tviz.show(tviz.plot_mode(blade_modes, mode=1))

## 7. Diving: speed and depth over a dive, pressure with depth, and what a propulsion loss does

A longitudinal model with what the earlier sections produced: the thrust curves of the drive
(section 4), the resistance curve (section 3), the reserve buoyancy and mass (section 2), and the
hull stress per unit pressure (section 5). Along the path: `m du/dt = T(u, throttle) − R(u) − N_B sin θ`
with `N_B` the reserve buoyancy force and `θ` the commanded dive angle (positive nose-down); depth
changes at `u sin θ`, plus a buoyant drift that the dive planes cancel while there is speed to work
with and that takes over when the propulsion stops. Added mass in surge and heave, and a crossflow drag
for the drift, are stated below. The dive planes are not modelled beyond "authority fades below
0.5 m/s"; pitch dynamics and currents are outside this model.

In [ ]:
P_ATM = 101325.0
HOTSPOT_PER_MPA = res.metrics["max_von_mises"] / P_DESIGN            # hull hotspot von Mises per MPa of external pressure (linear)
depths = np.linspace(0, 1.6 * DESIGN_DEPTH_M, 65)
p_gauge = RHO_W * G * depths / 1e6
table_depths = [0, 50, 100, 150, DESIGN_DEPTH_M, TEST_FACTOR * DESIGN_DEPTH_M]
pressure_table = pd.DataFrame({d: {"gauge_MPa": RHO_W * G * d / 1e6, "absolute_bar": (P_ATM + RHO_W * G * d) / 1e5, "hull_hotspot_MPa": HOTSPOT_PER_MPA * RHO_W * G * d / 1e6,
                                   "SF_yield": AL.yield_strength / (HOTSPOT_PER_MPA * RHO_W * G * d / 1e6) if d else float("inf"),
                                   "SF_buckling": p_buckle / (RHO_W * G * d / 1e6) if d else float("inf")} for d in table_depths}).T
pressure_table.index.name = "depth_m"
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(p_gauge, depths, label="gauge pressure"); ax[0].plot((P_ATM + RHO_W * G * depths) / 1e6, depths, "--", label="absolute pressure")
ax[0].axhline(DESIGN_DEPTH_M, color="#2e7d32", ls=":", label="rated depth"); ax[0].axhline(TEST_FACTOR * DESIGN_DEPTH_M, color="#c62828", ls=":", label="test depth")
ax[0].invert_yaxis(); ax[0].set(xlabel="pressure [MPa]", ylabel="depth [m]", title="pressure vs depth"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].plot(HOTSPOT_PER_MPA * p_gauge, depths, label="hull hotspot von Mises (FEA, linear)")
ax[1].axvline(AL.yield_strength, color="#c62828", ls="--", label=f"yield {AL.yield_strength:.0f} MPa")
ax[1].axhline(p_buckle * 1e6 / (RHO_W * G), color="#6a1b9a", ls="-.", label=f"elastic collapse depth {p_buckle * 1e6 / (RHO_W * G):.0f} m")
ax[1].axhline(DESIGN_DEPTH_M, color="#2e7d32", ls=":", label="rated depth")
ax[1].invert_yaxis(); ax[1].set(xlabel="stress [MPa]", ylabel="depth [m]", title="hull stress vs depth", xlim=(0, AL.yield_strength * 1.1)); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
fig.tight_layout()
pressure_table.round(2)

In [ ]:
M_SURGE = M_KG * 1.05                                     # surge added mass ~5 % for a slender body (assumed)
M_HEAVE = M_KG * 2.0                                      # heave added mass ~ the displaced mass for a long cylinder in crossflow (assumed)
CD_Z, A_Z = 1.0, L_M * D_M                                # crossflow drag coefficient and projected area of the hull
N_B = (buoyancy_kg - M_KG) * G                            # reserve buoyancy force [N], upward
AUTHORITY_SPEED = 0.5                                     # m/s below which the dive planes stop holding depth
u_grid = np.linspace(0.05, V_MAX * 1.05, 24)
T_grid = {thr: np.array([drive.at_throttle(thr, WAKE * u).thrust for u in u_grid]) * (1 - T_DED) for thr in (cruise.throttle, 1.0)}
def thrust(u, thr): return float(np.interp(u, u_grid, T_grid[thr])) if thr > 0 else 0.0

def simulate(phases, dt=1.0, fail_at=None, t_max=7200.0):
    """phases: (name, throttle, dive angle deg, ('depth', m) | ('time', s)); fail_at: time when the propulsion stops."""
    t, z, u, w_b = 0.0, 0.0, 0.5, 0.0
    rows, thrusting = [], True
    for name, thr, angle, (kind, target) in phases:
        t0 = t
        while t < t_max:
            if fail_at is not None and t >= fail_at:
                thrusting = False
            thr_eff = thr if thrusting else 0.0
            authority = min(1.0, u / AUTHORITY_SPEED)
            th = math.radians(angle * authority)
            drag = resistance(max(u, 0.05))[0]
            u = max(u + (thrust(u, thr_eff) - drag - N_B * math.sin(th)) / M_SURGE * dt, 0.0)
            w_b += (-N_B - 0.5 * RHO_W * CD_Z * A_Z * w_b * abs(w_b)) / M_HEAVE * dt    # buoyant drift (down positive)
            w_b *= (1 - authority)                                                       # the planes cancel it while they can
            w = u * math.sin(th) + w_b
            z = max(z + w * dt, 0.0)
            t += dt
            rows.append({"t_s": t, "phase": name, "depth_m": z, "speed_m_s": u, "vertical_m_s": w, "throttle": thr_eff, "pressure_MPa": RHO_W * G * z / 1e6})
            done = (kind == "time" and t - t0 >= target) or (kind == "depth" and ((angle > 0 and z >= target) or (angle <= 0 and z <= target)))
            if not thrusting and z <= 0.0:
                return pd.DataFrame(rows)
            if done and thrusting:
                break
    return pd.DataFrame(rows)

PROFILE = [("surface run", cruise.throttle, 0.0, ("time", 60)), ("dive", cruise.throttle, 15.0, ("depth", DESIGN_DEPTH_M)),
           ("level at rated depth", cruise.throttle, 0.0, ("time", 600)), ("climb", cruise.throttle, -15.0, ("depth", 5.0)), ("surface run", cruise.throttle, 0.0, ("time", 60))]
dive = simulate(PROFILE)
t_fail = float(dive[dive.phase == "level at rated depth"].t_s.iloc[0]) + 300.0
loss = simulate(PROFILE, fail_at=t_fail)
after = loss[loss.t_s >= t_fail]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(dive.t_s / 60, dive.depth_m, label="profile dive"); ax[0].plot(loss.t_s / 60, loss.depth_m, "--", label=f"propulsion lost at t = {t_fail / 60:.0f} min")
ax[0].invert_yaxis(); ax[0].set(xlabel="time [min]", ylabel="depth [m]"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].plot(dive.speed_m_s, dive.depth_m, label="profile dive"); ax[1].plot(after.speed_m_s, after.depth_m, "--", label="after the propulsion loss")
ax[1].invert_yaxis(); ax[1].set(xlabel="speed [m/s]", ylabel="depth [m]", title="speed vs depth"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
ax[2].plot(after.t_s - t_fail, after.speed_m_s, label="speed"); ax2 = ax[2].twinx(); ax2.plot(after.t_s - t_fail, after.depth_m, color="#c62828", label="depth"); ax2.invert_yaxis()
ax[2].set(xlabel="time after the propulsion loss [s]", ylabel="speed [m/s]"); ax2.set_ylabel("depth [m]", color="#c62828"); ax[2].grid(alpha=0.3); ax[2].set_title("propulsion loss: speed decays, the reserve buoyancy brings it up")
fig.tight_layout()
print(f"profile dive: {dive.t_s.iloc[-1] / 60:.0f} min; speed at the bottom of the dive {dive[dive.phase == 'dive'].speed_m_s.iloc[-1]:.2f} m/s vs {V_CRUISE} m/s level "
      f"(the reserve buoyancy {N_B:.1f} N works against a {PROFILE[1][2]:.0f} deg dive)")
print(f"propulsion loss at {loss[loss.t_s >= t_fail].depth_m.iloc[0]:.0f} m: speed below 0.1 m/s after {float(after[after.speed_m_s < 0.1].t_s.iloc[0] - t_fail) if (after.speed_m_s < 0.1).any() else float('nan'):.0f} s, "
      f"terminal rise {abs(after.vertical_m_s.min()):.3f} m/s, at the surface after {(after.t_s.iloc[-1] - t_fail) / 60:.0f} min")
pd.DataFrame({f"+{s} s": {"speed_m_s": float(np.interp(t_fail + s, after.t_s, after.speed_m_s)), "depth_m": float(np.interp(t_fail + s, after.t_s, after.depth_m)),
                          "pressure_MPa": float(np.interp(t_fail + s, after.t_s, after.pressure_MPa))} for s in (0, 30, 60, 120, 300, 600, 1200)}).T.round(3)

## 8. Depth cycling (Chronos)

Each dive is a pressure cycle on the hull. Three dive profiles (the load level is the external pressure
in MPa), rainflow-counted into spectra; the stress per unit pressure at the FEA hotspot and a Basquin
curve for 6061-T6 give the damage per dive. There is no vibratory excitation here, so no structure
dynamics enter the spectra.

In [ ]:
def depth(m): return RHO_W * G * m / 1e6
missions = {
    "shallow survey": chronos.Mission("shallow survey", (
        chronos.Segment("descent", 300, {"pressure_MPa": depth(30)}),
        chronos.Segment("terrain following", 300, {"pressure_MPa": depth(45)}, repeat=40, notes="excursions from 30 m to 45 m"),
        chronos.Segment("survey lines", 7200, {"pressure_MPa": depth(30)}),
        chronos.Segment("ascent", 300, {"pressure_MPa": depth(2)}),
    ), "4 h at 30 m with forty 15 m excursions"),
    "yo-yo profiling": chronos.Mission("yo-yo profiling", (
        chronos.Segment("at the surface", 120, {"pressure_MPa": depth(2)}),
        chronos.Segment("profile to 150 m", 600, {"pressure_MPa": depth(150)}, repeat=25, notes="25 profiles between 2 m and 150 m"),
        chronos.Segment("recovery", 300, {"pressure_MPa": depth(2)}),
    ), "25 profiles to 150 m"),
    "deep station": chronos.Mission("deep station", (
        chronos.Segment("descent", 900, {"pressure_MPa": depth(DESIGN_DEPTH_M)}),
        chronos.Segment("station keeping at rated depth", 6 * 3600, {"pressure_MPa": depth(DESIGN_DEPTH_M)}),
        chronos.Segment("ascent", 900, {"pressure_MPa": depth(2)}),
    ), "one dive to the rated depth, 6 h on station"),
}
spectra = {k: chronos.build_spectrum(m) for k, m in missions.items()}
for k, sp in spectra.items():
    sp.save(RUNS / f"spectrum_{k.replace(' ', '_')}.json")
fig, axes = plt.subplots(3, 1, figsize=(10, 7.5), sharex=False)
for ax, (k, m) in zip(axes, missions.items()):
    m.profile(ax=ax); ax.set_title(k)
fig.tight_layout()
SN_AL = chronos.SNCurve("6061-T6 (Basquin, handbook)", sigma_f=535.0, b=-0.09, ultimate=310.0)
stress_per_mpa = {"pressure_MPa": HOTSPOT_PER_MPA}                              # hotspot von Mises per MPa of external pressure (section 7)
damage = {k: chronos.hotspot_damage(sp, stress_per_mpa, SN_AL)["total"] for k, sp in spectra.items()}
pd.DataFrame({k: {"duration_h": m.duration_h, "pressure_cycles": spectra[k].total_cycles, "peak_hotspot_MPa": max(s.loads.get("pressure_MPa", 0) for s in m.segments) * stress_per_mpa["pressure_MPa"],
                  "damage_per_dive": damage[k], "dives_to_failure": (1 / damage[k]) if damage[k] > 0 else float("inf")} for k, m in missions.items()}).T

## 9. Export

In [ ]:
doc = {"design": {"file": "designs/submarine.py", "parameters": p, "pressure_hull_offset_mm": X_PH},
       "volumes_L": {"vehicle": mv["volume"] * 1e-6, "body": mb["volume"] * 1e-6, "pressure_hull_shell": mh["volume"] * 1e-6},
       "wetted_surface_m2": S_WET, "mass_budget": items.to_dict(orient="index"), "hydrostatics": {k: float(v) for k, v in hydro.items()}, "cg_mm": CG.tolist(), "cb_mm": CB.tolist(),
       "resistance_N": dict(zip([round(v, 2) for v in Vs], R[:, 0].round(3).tolist())),
       "cfd_cruise": ({"parameters": CFD_PARAMS, "metrics": {k: v for k, v in cfd.metrics.items() if not isinstance(v, (list, dict))}} if cfd is not None and cfd.ok else "not run"),
       "propulsion": {"propeller": prop.describe(), "wake_fraction": WAKE, "thrust_deduction": T_DED, "hotel_W": HOTEL_W, "top_speed_m_s": V_MAX,
                      "cruise": cruise.to_dict(), "full": full.to_dict(), "range_km_vs_speed": dict(zip(speeds.round(2).tolist(), range_km.round(1).tolist()))},
       "pressure_hull": {"design_depth_m": DESIGN_DEPTH_M, "design_pressure_MPa": P_DESIGN, "material": AL.__dict__, "wet_faces": wet_tags,
                         "fea": {k: v for k, v in res.metrics.items() if not isinstance(v, (dict, list))}, "theory": check["thin-shell theory"].to_dict(), "collapse": collapse.to_dict()},
       "propeller_cad": CAD_KW, "propeller_cfd_cruise": ({"parameters": PROP_CFD, "metrics": {k: v for k, v in cfd_prop.metrics.items() if not isinstance(v, (list, dict))}} if cfd_prop is not None and cfd_prop.ok else "not run"),
       "noise": noise.to_dict(orient="index"), "blade_fea_top_speed": {k: v for k, v in res_blade.metrics.items() if not isinstance(v, (dict, list))},
       "blade_modes_hz_air": f_air, "blade_modes_hz_water_assumed": f_water, "hull_ring_mode_hz": f_ring,
       "pressure_vs_depth": pressure_table.to_dict(orient="index"), "dive": {"profile": [list(ph[:3]) + [list(ph[3])] for ph in PROFILE], "duration_min": dive.t_s.iloc[-1] / 60,
                "propulsion_loss": {"t_fail_s": t_fail, "time_to_surface_min": (after.t_s.iloc[-1] - t_fail) / 60, "terminal_rise_m_s": abs(after.vertical_m_s.min())}},
       "missions": {k: m.describe() for k, m in missions.items()}, "damage_per_dive": damage, "sn_curve": SN_AL.__dict__}
(RUNS / "submarine.json").write_text(json.dumps(doc, indent=2, default=float))
print("written:", sorted(x.name for x in RUNS.iterdir()))

**Reading the result:** the mass budget says where the trim lead must go and how much stability lever
the vehicle has; the resistance curve and the CFD check (run it) fix the power at cruise and the
endurance; the pressure hull has its margins to yield and to elastic collapse at the rated depth, with
the FEA agreeing with thin-shell theory away from the ends; the propeller check says how far blade
element theory is off for this planform, what a hydrophone hears, and that the blade modes clear the
rotor lines; the dive simulation shows the speed the reserve buoyancy costs on the way down and how the
vehicle comes back up on its own when the propulsion stops; and depth cycling is shown not to be a
fatigue driver for the shell itself — the penetrator seals, welds and corrosion, which this model does
not contain, are what limit a real hull.